# Module 12 - 실습 4: astream 토큰 스트리밍

## 학습 목표
- LangGraph의 `astream` 메서드로 실시간 스트리밍을 구현할 수 있다
- `updates` 모드로 노드 단위 업데이트를 수신할 수 있다
- FakeLLM과 스트리밍을 함께 사용할 수 있다

## 참조 자료
- 학습 문서: `docs/part5-advanced/12-langgraph-advanced.md` (섹션 2.4)

> **Jupyter 주의**: Jupyter에서는 `await`를 직접 사용합니다 (`asyncio.run()` 대신).

## 개념 요약

**스트리밍 모드 비교**:
| 모드 | 내용 | 용도 |
|------|------|------|
| `"values"` | 각 스텝 후 전체 상태 | 상태 변화 추적 |
| `"updates"` | 노드가 반환한 변경분만 | 경량 업데이트 |
| `"messages"` | LLM 토큰 + 메타데이터 | 실시간 텍스트 출력 |

**기본 패턴**:
```python
# Jupyter에서 직접 await 사용
async for event in app.astream(input_state, config=config, stream_mode="updates"):
    for node_name, updates in event.items():
        print(f"[{node_name}] 완료")
```

In [ ]:
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from common.fake_llm import FakeLLM

print("스트리밍 실습 준비 완료")

## 실습 1: 스트리밍 그래프 구성

research → analyze → answer 3단계 그래프를 구성하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: StreamState에 query, research, analysis, final_answer 필드
# 힌트 2: 각 노드는 해당 필드만 업데이트
# 힌트 3: FakeLLM으로 분석 수행

class StreamState(TypedDict):
    query: str
    research: str | None
    analysis: str | None
    final_answer: str | None

fake_llm = FakeLLM(responses={
    "default": "분석 결과: Python 비동기 프로그래밍은 asyncio 라이브러리를 사용합니다."
})

def research_node(state: StreamState) -> dict:
    """리서치를 수행합니다."""
    # TODO: query를 참조하여 research 결과 생성
    pass

def analyze_node(state: StreamState) -> dict:
    """LLM으로 분석을 수행합니다."""
    # TODO: FakeLLM으로 research 결과를 분석
    pass

def answer_node(state: StreamState) -> dict:
    """최종 답변을 생성합니다."""
    # TODO: analysis를 기반으로 final_answer 생성
    pass

# 그래프 구성
graph = StateGraph(StreamState)
# TODO: 노드 추가, 엣지 연결

checkpointer = MemorySaver()
# app = graph.compile(checkpointer=checkpointer)

## 실습 2: updates 모드 스트리밍

`stream_mode="updates"`로 각 노드 완료 시 변경된 필드를 수신하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: app.astream(input_state, config=config, stream_mode="updates") 사용
# 힌트 2: async for event in app.astream(...) 로 이벤트 수신
# 힌트 3: event는 {노드이름: {변경된 필드들}} 형태

config = {"configurable": {"thread_id": "stream-001"}}
input_state = {
    "query": "Python 비동기 프로그래밍의 장단점",
    "research": None,
    "analysis": None,
    "final_answer": None,
}

print("=== 노드 업데이트 스트리밍 ===\n")
received_nodes = []

# TODO: async for 루프로 스트리밍
# async for event in app.astream(...):
#     for node_name, updates in event.items():
#         received_nodes.append(node_name)
#         print(f"[{node_name}] 완료")
#         ...

print(f"\n수신한 노드 순서: {received_nodes}")

In [ ]:
# 검증 셀
assert len(received_nodes) >= 3, f"최소 3개 노드 업데이트를 수신해야 합니다. 현재: {received_nodes}"
assert "research" in received_nodes, "research 노드 이벤트를 수신해야 합니다"
assert "analyze" in received_nodes, "analyze 노드 이벤트를 수신해야 합니다"
assert "answer" in received_nodes, "answer 노드 이벤트를 수신해야 합니다"
print("✅ 실습 2 완료! 노드 단위 스트리밍이 올바르게 작동합니다.")

## 실습 3: values 모드 스트리밍

`stream_mode="values"`로 각 스텝 후 전체 상태를 수신하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: stream_mode="values" 로 전체 상태 스트리밍
# 힌트 2: event는 전체 상태 딕셔너리 (노드명 키 없음)
# 힌트 3: 다른 thread_id 사용

config2 = {"configurable": {"thread_id": "stream-002"}}
input_state2 = {
    "query": "asyncio vs threading 비교",
    "research": None,
    "analysis": None,
    "final_answer": None,
}

print("=== values 모드 스트리밍 ===\n")
state_snapshots = []

# TODO: stream_mode="values"로 스트리밍
# async for state in app.astream(input_state2, config=config2, stream_mode="values"):
#     state_snapshots.append(dict(state))
#     print(f"현재 상태: research={state.get('research') is not None}, analysis={state.get('analysis') is not None}")

print(f"\n총 {len(state_snapshots)}번 상태 업데이트 수신")

In [ ]:
# 검증 셀
assert len(state_snapshots) >= 3, f"최소 3번의 상태 업데이트가 있어야 합니다. 현재: {len(state_snapshots)}"
# 마지막 상태에는 모든 필드가 채워져야 함
final_state = state_snapshots[-1]
assert final_state.get("research") is not None, "최종 상태에 research가 없습니다"
assert final_state.get("analysis") is not None, "최종 상태에 analysis가 없습니다"
assert final_state.get("final_answer") is not None, "최종 상태에 final_answer가 없습니다"
print("✅ 실습 3 완료! values 모드 스트리밍이 올바르게 작동합니다.")

## 정리

이번 실습에서 배운 내용:
1. `astream()`은 `invoke()`의 비동기 스트리밍 버전이다
2. `updates` 모드는 노드가 변경한 필드만, `values` 모드는 전체 상태를 반환한다
3. Jupyter에서는 `async for`를 셀에서 직접 사용한다

## 축하합니다!
Module 12까지 모든 실습을 완료했습니다! LangGraph의 핵심 고급 패턴을 모두 익혔습니다.